## 1. Import Libraries

We import `pandas` and inject our project root into `sys.path` so we can natively import our custom `DataNormalizer` module from the ETL pipeline.

In [ ]:
import pandas as pd
from pathlib import Path
import sys

current_dir = Path.cwd()

# Dynamically resolve the project root
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Allow imports from 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl.normalizer import DataNormalizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("Libraries Imported Successfully")

## 2. Project Paths

We configure dynamic paths to automatically find our `raw` data, `supporting` data, and establish a new `processed` data directory for normalized files.

In [ ]:
RAW_DATA = PROJECT_ROOT / "data" / "raw"
SUPPORTING_DATA = PROJECT_ROOT / "data" / "supporting"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

# Ensure processed data folder exists
PROCESSED_DATA.mkdir(exist_ok=True, parents=True)

print("Project Root:", PROJECT_ROOT)

## 3. Find All Datasets

We recursively collect all Excel files from both the raw and supporting data directories to build our processing queue.

In [ ]:
raw_files = sorted(RAW_DATA.glob("*.xlsx"))
supporting_files = sorted(SUPPORTING_DATA.glob("*.xlsx"))

excel_files = raw_files + supporting_files

print(f"Datasets Found : {len(excel_files)}")

for file in excel_files:
    print(file.name)

## 4. Test One Dataset

We first load the master `companies.xlsx` file to test the normalizer and review its initial data types and structure.

In [ ]:
try:
    df = pd.read_excel(RAW_DATA / "companies.xlsx")
    display(df.head())
    print("\n--- Dtypes ---")
    print(df.dtypes)
except Exception as e:
    print(f"Error reading file: {e}")

## 5. Apply Normalizer

We run the dataframe through our `DataNormalizer` pipeline, scrubbing bad characters, casting types, and replacing string nulls with `NaN`.

In [ ]:
if 'df' in locals():
    normalized_df = DataNormalizer.normalize(df.copy())
    display(normalized_df.head())

## 6. Compare Before & After

We compare the data types before and after normalization to ensure our pipeline successfully converted strings to numerics and integers where appropriate.

In [ ]:
if 'df' in locals() and 'normalized_df' in locals():
    comparison = pd.DataFrame({
        "Column": df.columns,
        "Before (Dtype)": df.dtypes.astype(str).values,
        "After (Dtype)": normalized_df.dtypes.astype(str).values
    })
    display(comparison)

## 7. Check Missing Values

We verify that our normalizer correctly identified ghost-nulls (like `"-"`, `"NA"`) and converted them to standard pandas nulls.

In [ ]:
if 'df' in locals() and 'normalized_df' in locals():
    before = df.isnull().sum()
    after = normalized_df.isnull().sum()

    missing = pd.DataFrame({
        "Column": df.columns,
        "Nulls Before": before.values,
        "Nulls After": after.values
    })
    display(missing)

## 8. Normalize Every Dataset

We loop through all discovered Excel files, normalize them completely, and save the scrubbed output into the `processed` data folder as CSV files.

In [ ]:
summary = []

for file in excel_files:
    try:
        print("=" * 80)
        print(f"Processing: {file.name}")

        # Extract
        temp_df = pd.read_excel(file)
        rows_before = len(temp_df)

        # Transform / Normalize
        temp_df = DataNormalizer.normalize(temp_df)
        rows_after = len(temp_df)

        # Load / Save
        output = PROCESSED_DATA / f"{file.stem}.csv"
        temp_df.to_csv(output, index=False)

        # Compile metrics
        summary.append({
            "Dataset": file.name,
            "Rows": rows_after,
            "Columns": len(temp_df.columns),
            "Missing Values": int(temp_df.isnull().sum().sum()),
            "Output": output.name
        })

        print(f"Saved -> {output.name}")
    except Exception as e:
        print(f"Failed to process {file.name}: {e}")

## 9. Summary Report

We assemble the processing metrics into a unified dataframe for review.

In [ ]:
summary_df = pd.DataFrame(summary)
display(summary_df)

## 10. Export Report

We export the normalization report to the `reports` directory so the data quality team can review it.

In [ ]:
REPORT_PATH = PROJECT_ROOT / "reports"
REPORT_PATH.mkdir(exist_ok=True)

if not summary_df.empty:
    summary_df.to_csv(
        REPORT_PATH / "normalization_report.csv",
        index=False
    )
    print("Normalization Report Saved Successfully")
else:
    print("No data to save.")

## 11. Dataset Statistics

We print out aggregate statistics representing the scale and cleanliness of the processed data warehouse.

In [ ]:
print("=" * 70)
print("NORMALIZATION SUMMARY")
print("=" * 70)

if not summary_df.empty:
    print(f"Datasets : {len(summary_df)}")
    print(f"Total Rows : {summary_df['Rows'].sum()}")
    print(f"Total Columns : {summary_df['Columns'].sum()}")
    print(f"Total Missing Values : {summary_df['Missing Values'].sum()}")

## 12. Preview Processed Files

We confirm the existence of all successfully normalized and exported CSV files inside our `processed` data directory.

In [ ]:
processed_files = sorted(PROCESSED_DATA.glob("*.csv"))

print(f"Processed Files : {len(processed_files)}")

for file in processed_files:
    print(file.name)